# Schwellenwertvergleich — P95 vs P99.5
**je Gebäudetyp** *(optionales Referenz-Notebook, unnummeriert wie
`clustering_alternative.ipynb` im NL-Repositorium)*

P95 wurde als Alternative zu P99.5 für die Ausreißer-Entfernung getestet. Die
Sorge: große Ausreißer-Gebäude (vermutlich falsch klassifizierte Industrie-/
Hofgebäude) belegen einen ganzen Cluster und verschlechtern die Genauigkeit für
die restlichen 99 %.

**Aktueller Ansatz (P99.5):** entfernt die oberen 0,5 % je Gebäudetyp — ca. 20.700 Gebäude
**P95-Ansatz:** entfernt die oberen 5 % je Gebäudetyp — aggressiver

Dieses Notebook vergleicht:
- wie viele Gebäude je Schwelle entfernt werden,
- ob sich die Clustering-Ergebnisse sinnvoll ändern,
- ob der große Ausreißer-Cluster bei P95 verschwindet.

**Hinweis zur Methode:** Das Vergleichs-Clustering nutzt denselben Log-Raum-KMeans
mit Coverage-Kriterium wie `02_nrw_clustering.ipynb`.

## 1. Konfiguration

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE_DIR      = "."
FILE_GEBAEUDE = os.path.join(BASE_DIR, "nrw_buildings.parquet")
OUTPUT_DIR    = os.path.join(BASE_DIR, "nrw_results", "schwellenwertvergleich")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SIZE_CLASSES = ["SFH", "TH", "MFH", "AB"]

# Clustering-Einstellungen — identisch zu 02_nrw_clustering.ipynb
K_MAX = 10
KMEANS_1D_MAX_ITER = 10_000
COVERAGE_MIN = 0.80
TOLERANZ_BAND = 0.20
MIN_GEBAEUDE_FUER_CLUSTERING = 50

print("Bereit")

## 2. Daten laden

In [ ]:
gebaeude = pd.read_parquet(FILE_GEBAEUDE)
print(f'Gebaeude geladen: {len(gebaeude):,}')
print("\nZahl je Gebaeudetyp:")
print(gebaeude["size_class"].value_counts().to_string())

## 3. Entfernung bei verschiedenen Schwellen vergleichen

Prüft, wie viele Gebäude je Gebäudetyp bei P90, P95, P99, P99.5 entfernt
werden — zeigt, wo die Ausreißer konzentriert sind.

In [ ]:
schwellen = [0.90, 0.95, 0.99, 0.995]

print("Je Schwelle entfernte Gebaeude:")
print(f'  {"Schwelle":<12} {"SFH":>10} {"TH":>10} {"MFH":>10} {"AB":>10} {"GESAMT":>10}')
print("-" * 65)

for thr in schwellen:
    entfernt_je_sc = {}
    for sc in SIZE_CLASSES:
        maske = gebaeude["size_class"] == sc
        p = gebaeude.loc[maske, "area_m2"].quantile(thr)
        entfernt_je_sc[sc] = int((gebaeude.loc[maske, "area_m2"] > p).sum())
    gesamt = sum(entfernt_je_sc.values())
    anteil = gesamt / len(gebaeude) * 100
    print(f'  P{thr*100:<9.1f} '
          f'{entfernt_je_sc["SFH"]:>10,} '
          f'{entfernt_je_sc["TH"]:>10,} '
          f'{entfernt_je_sc["MFH"]:>10,} '
          f'{entfernt_je_sc["AB"]:>10,} '
          f'{gesamt:>10,}  ({anteil:.2f}%)')

## 4. Beide Schwellen anwenden und Verteilungen vergleichen

P99.5 und P95 getrennt anwenden und die resultierenden
Wärmebedarfsverteilungen vergleichen. Wenn P95 den Ausreißer-Cluster
entfernt, sollte die obere Ränder der Verteilung abgeschnitten sein.

In [ ]:
def schwelle_anwenden(df, quantil):
    behalten = pd.Series(True, index=df.index)
    schwellen_werte = {}
    for sc in SIZE_CLASSES:
        maske = df["size_class"] == sc
        p = df.loc[maske, "area_m2"].quantile(quantil)
        schwellen_werte[sc] = p
        behalten &= ~((df["size_class"] == sc) & (df["area_m2"] > p))
    return df[behalten].copy(), schwellen_werte

daten_p995, werte_p995 = schwelle_anwenden(gebaeude, 0.995)
daten_p95,  werte_p95  = schwelle_anwenden(gebaeude, 0.95)

print(f'P99.5 behalten: {len(daten_p995):,}  (entfernt {len(gebaeude)-len(daten_p995):,}  '
      f'= {(len(gebaeude)-len(daten_p995))/len(gebaeude)*100:.2f}%)')
print(f'P95   behalten: {len(daten_p95):,}  (entfernt {len(gebaeude)-len(daten_p95):,}  '
      f'= {(len(gebaeude)-len(daten_p95))/len(gebaeude)*100:.2f}%)')

print("\nFlaechen-Schwellen (m2):")
print(f'  {"Typ":<6} {"P99.5":>10} {"P95":>10}  Differenz')
for sc in SIZE_CLASSES:
    diff = werte_p995[sc] - werte_p95[sc]
    print(f'  {sc:<6} {werte_p995[sc]:>10.1f} {werte_p95[sc]:>10.1f}  -{diff:.1f} m2')

In [ ]:
# Waermebedarfsverteilungen vergleichen
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, sc in enumerate(SIZE_CLASSES):
    sub_p995 = daten_p995[daten_p995["size_class"] == sc]["heat_total_kwh"]
    sub_p95  = daten_p95[daten_p95["size_class"] == sc]["heat_total_kwh"]

    cap = sub_p995.quantile(0.99)

    # Obere Reihe: Histogramm-Vergleich
    ax = axes[0][col]
    ax.hist(sub_p995.clip(upper=cap), bins=50, alpha=0.6,
            color="steelblue", label=f"P99.5  n={len(sub_p995):,}")
    ax.hist(sub_p95.clip(upper=cap),  bins=50, alpha=0.6,
            color="tomato",   label=f"P95    n={len(sub_p95):,}")
    ax.set_title(f"{sc} — Waermebedarfsverteilung", fontsize=9, fontweight="bold")
    ax.set_xlabel("Waermebedarf (kWh/Jahr)", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Untere Reihe: Boxplot-Vergleich
    ax2 = axes[1][col]
    bp = ax2.boxplot([sub_p995.values, sub_p95.values],
                     patch_artist=True,
                     medianprops={"color": "black", "linewidth": 2})
    bp["boxes"][0].set_facecolor("steelblue"); bp["boxes"][0].set_alpha(0.7)
    bp["boxes"][1].set_facecolor("tomato");    bp["boxes"][1].set_alpha(0.7)
    ax2.set_xticklabels(["P99.5", "P95"], fontsize=9)
    ax2.set_title(f"{sc} — Boxplot-Vergleich", fontsize=9, fontweight="bold")
    ax2.set_ylabel("Waermebedarf (kWh/Jahr)", fontsize=8)
    ax2.grid(True, axis="y", alpha=0.3)

plt.suptitle("Waermebedarfsverteilung: P99.5 vs P95 Ausreisser-Entfernung",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "schwellenvergleich_verteilungen.pdf"),
            bbox_inches="tight")
print("Plot gespeichert")

## 5. Clustering-Ergebnisse vergleichen

Log-Raum-KMeans mit Coverage-Kriterium (identisch zu
`02_nrw_clustering.ipynb`) auf den Wärmebedarf beider Schwellen — prüft, ob
sich k und die Cluster-Zentren sinnvoll verschieben.

In [ ]:
# --- Log-Raum-KMeans (identisch zu 02_nrw_clustering.ipynb, Abschnitt 5) ---
def _kmeans_1d(values, k, n_iter=KMEANS_1D_MAX_ITER):
    log_values = np.log(values)
    log_centers = np.quantile(log_values, np.linspace(0, 1, k + 2)[1:-1])
    for _ in range(n_iter):
        distances = np.abs(log_values[:, None] - log_centers[None, :])
        labels = distances.argmin(axis=1)
        new_log_centers = np.array([
            log_values[labels == i].mean() if np.any(labels == i) else log_centers[i]
            for i in range(k)
        ])
        if np.allclose(new_log_centers, log_centers):
            break
        log_centers = new_log_centers
    return labels, np.exp(log_centers)

def kmeans_mit_abweichungskriterium(values, max_k=K_MAX,
                                    min_coverage=COVERAGE_MIN,
                                    max_deviation=TOLERANZ_BAND):
    best = None
    for k in range(1, max_k + 1):
        labels, centers = _kmeans_1d(values, k)
        rel_dev = np.abs(values - centers[labels]) / np.abs(centers[labels])
        share_within = np.mean(rel_dev <= max_deviation)
        if best is None or share_within > best[3]:
            best = (labels, centers, k, share_within)
        if share_within >= min_coverage:
            return labels, centers, k, share_within
    return best

print("Clustering mit P99.5...")
ergebnisse_p995 = {}
for sc in SIZE_CLASSES:
    werte = daten_p995.loc[daten_p995["size_class"] == sc, "heat_total_kwh"].values.astype(float)
    werte = werte[werte > 0]
    labels, zentren, k, coverage = kmeans_mit_abweichungskriterium(werte)
    ergebnisse_p995[sc] = {"k": k, "zentren": zentren, "coverage": coverage}

print("Clustering mit P95...")
ergebnisse_p95 = {}
for sc in SIZE_CLASSES:
    werte = daten_p95.loc[daten_p95["size_class"] == sc, "heat_total_kwh"].values.astype(float)
    werte = werte[werte > 0]
    labels, zentren, k, coverage = kmeans_mit_abweichungskriterium(werte)
    ergebnisse_p95[sc] = {"k": k, "zentren": zentren, "coverage": coverage}

print("\nZENTREN-VERGLEICH (Waermebedarf kWh/Jahr):")
print(f'  {"Typ":<6} {"P99.5":<28} {"P95":<28}')
print("-" * 64)
for sc in SIZE_CLASSES:
    z_a = " ".join(f"{z:,.0f}" for z in ergebnisse_p995[sc]["zentren"])
    z_b = " ".join(f"{z:,.0f}" for z in ergebnisse_p95[sc]["zentren"])
    print(f'  {sc:<6} k={ergebnisse_p995[sc]["k"]} [{z_a}]')
    print(f'  {"":<6} k={ergebnisse_p95[sc]["k"]} [{z_b}]')
    print(f'  {"":<6} Coverage P99.5: {ergebnisse_p995[sc]["coverage"]:.0%}  |  '
          f'P95: {ergebnisse_p95[sc]["coverage"]:.0%}')

## 6. Zusammenfassung und Empfehlung

In [ ]:
print("ZUSAMMENFASSUNG")
print("="*60)
print()
print(f'  P99.5 behalten: {len(daten_p995):,} Gebaeude ({len(daten_p995)/len(gebaeude)*100:.1f}%)')
print(f'  P95   behalten: {len(daten_p95):,} Gebaeude ({len(daten_p95)/len(gebaeude)*100:.1f}%)')
print()

k_veraenderungen = sum(1 for sc in SIZE_CLASSES
                      if ergebnisse_p995[sc]["k"] != ergebnisse_p95[sc]["k"])
print(f'  k-Werte veraendert (P99.5 -> P95): {k_veraenderungen} von {len(SIZE_CLASSES)}')
print()
print("  EMPFEHLUNG: P95 entfernt zehnmal so viele Gebaeude wie P99.5 und wurde")
print("  verworfen, weil es zu viele echte Gebaeude entfernt.")
print("  Im Coverage-Kriterium richtet sich k danach, wie viele Gebaeude")
print(f"  innerhalb +/-{TOLERANZ_BAND:.0%} ihres Zentrums liegen.")
print()
print(f"  Ergebnisse gespeichert unter: nrw_results/schwellenwertvergleich/")